In [1]:
!pip install pytorch-crf

In [2]:
import pandas as pd
import numpy as np
import re
import pickle as pkl
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn
from tqdm import tqdm
import fasttext
from torchcrf import CRF

In [3]:
train_df=pd.read_parquet("/kaggle/input/datasets/karimmahmoud09/pii-splited-data/train.parquet")
val_df=pd.read_parquet("/kaggle/input/datasets/karimmahmoud09/pii-splited-data/val.parquet")
test_df=pd.read_parquet("/kaggle/input/datasets/karimmahmoud09/pii-splited-data/test.parquet")

In [4]:
train_df.head()["tokenised_unmasked_text"]

0    [dear, hr, ,, please, initiate, the, process, ...
1    [notre, savings, account, doi, ##t, se, confor...
2    [", our, office, ,, located, at, 70, ##7, ##29...
3    [no, ##us, aim, ##eri, ##ons, organise, ##r, u...
4    [to, ensure, we, remain, connected, throughout...
Name: tokenised_unmasked_text, dtype: object

In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17330 entries, 0 to 17329
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   masked_text              17330 non-null  object
 1   unmasked_text            17330 non-null  object
 2   token_entity_labels      17330 non-null  object
 3   tokenised_unmasked_text  17330 non-null  object
dtypes: object(4)
memory usage: 541.7+ KB


In [6]:
print(len(train_df), len(val_df),len(test_df)) 

17330 1926 2140


In [7]:
ENWE = fasttext.load_model("/kaggle/input/datasets/bobazooba/fasttext-english/cc.en.300.bin")

In [8]:
all_labels = set()

for labels in train_df["token_entity_labels"]:
    all_labels.update(labels)

label2id = {label: idx for idx, label in enumerate(sorted(all_labels))}
id2label = {idx: label for label, idx in label2id.items()}

print(label2id)
print("Number of classes:", len(label2id))

{'B-ACCOUNTNAME': 0, 'B-ACCOUNTNUMBER': 1, 'B-CREDITCARDNUMBER': 2, 'B-EMAIL': 3, 'B-IPV4': 4, 'B-IPV6': 5, 'B-MAC': 6, 'B-PASSWORD': 7, 'B-PHONE_NUMBER': 8, 'B-SSN': 9, 'B-USERNAME': 10, 'I-ACCOUNTNAME': 11, 'I-ACCOUNTNUMBER': 12, 'I-CREDITCARDNUMBER': 13, 'I-EMAIL': 14, 'I-IPV4': 15, 'I-IPV6': 16, 'I-MAC': 17, 'I-PASSWORD': 18, 'I-PHONE_NUMBER': 19, 'I-SSN': 20, 'I-USERNAME': 21, 'O': 22}
Number of classes: 23


In [9]:
class PiiDataset(Dataset):

    def __init__(self, texts, labels, embedding_model, label2id):
        self.embedding_model = embedding_model
        self.label2id = label2id

        # ❗ FILTER OUT EMPTY SAMPLES (IMPORTANT FIX)
        self.data = [
            (t, l) for t, l in zip(texts, labels)
            if len(t) > 0 and len(l) > 0
        ]

        self.max_len = max(len(sent) for sent, _ in self.data)

        self.embed_dim = embedding_model.get_dimension()
        self.pad_embedding = torch.zeros(self.embed_dim)

        self.pad_label = -100

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        words, labels = self.data[idx]

        embeddings = [
            torch.tensor(
                self.embedding_model.get_word_vector(word),
                dtype=torch.float32
            )
            for word in words
        ]

        encoded_labels = [
            self.label2id[label]
            for label in labels
        ]

        length = len(words)

        # padding
        while len(embeddings) < self.max_len:
            embeddings.append(self.pad_embedding)

        while len(encoded_labels) < self.max_len:
            encoded_labels.append(self.pad_label)

        x = torch.stack(embeddings)

        y = torch.tensor(encoded_labels, dtype=torch.long)

        return x, length, y

In [10]:
training_data = PiiDataset(
    np.array(train_df["tokenised_unmasked_text"]),
    np.array(train_df["token_entity_labels"]),
    embedding_model=ENWE,
    label2id=label2id
)

val_data = PiiDataset(
    np.array(val_df["tokenised_unmasked_text"]),
    np.array(val_df["token_entity_labels"]),
    embedding_model=ENWE,
    label2id=label2id
)

test_data = PiiDataset(
    np.array(test_df["tokenised_unmasked_text"]),
    np.array(test_df["token_entity_labels"]),
    embedding_model=ENWE,
    label2id=label2id
)

In [11]:
train_loader = DataLoader(
    training_data,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False
)

In [12]:
from torchcrf import CRF
import torch
import torch.nn as nn


class BiLSTMCRF(nn.Module):

    def __init__(
        self,
        embedding_dim=300,
        hidden_size=256,
        num_classes=2,
        dropout=0.3
    ):
        super().__init__()

        self.bilstm1 = nn.LSTM(
            embedding_dim,
            hidden_size,
            batch_first=True,
            bidirectional=True
        )

        self.bilstm2 = nn.LSTM(
            hidden_size * 2,
            hidden_size,
            batch_first=True,
            bidirectional=True
        )

        self.bilstm3 = nn.LSTM(
            hidden_size * 2,
            hidden_size,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

        self.crf = CRF(
            num_tags=num_classes,
            batch_first=True
        )

    def forward(self, x, lengths):

        out1, _ = self.bilstm1(x)
        out2, _ = self.bilstm2(out1)
        out3, _ = self.bilstm3(out2)

        emissions = self.classifier(out3)

        return emissions

    def loss(self, x, lengths, tags, mask):

        emissions = self.forward(x, lengths)

        loss = -self.crf(
            emissions,
            tags,
            mask=mask,
            reduction="mean"
        )

        return loss

    def decode(self, x, lengths, mask):

        emissions = self.forward(x, lengths)

        prediction = self.crf.decode(
            emissions,
            mask=mask
        )

        return prediction

In [13]:
def overfit_one_batch(
    model,
    dataset,
    batch_size=1,
    epochs=500,
    lr=1e-3
):

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    x, lengths, y = next(iter(loader))

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model = model.to(device)

    x = x.to(device)
    y = y.to(device)

    lengths = lengths.to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    model.train()

    mask = (y != -100).bool()

    safe_labels = y.clone()
    safe_labels[~mask] = 0

    for epoch in range(epochs):

        optimizer.zero_grad()

        loss = model.loss(
            x,
            lengths,
            safe_labels,
            mask
        )

        loss.backward()

        optimizer.step()

        with torch.no_grad():

            preds = model.decode(
                x,
                lengths,
                mask
            )

            correct = 0
            total = 0

            for i in range(len(preds)):

                pred = torch.tensor(
                    preds[i],
                    device=device
                )

                true = safe_labels[i][mask[i]]

                correct += (pred == true).sum().item()
                total += len(true)

            acc = correct / total

            unique_preds = set()

            for seq in preds:
                unique_preds.update(seq)

        if epoch % 10 == 0:

            print(
                f"Epoch {epoch} | "
                f"Loss: {loss.item():.4f} | "
                f"Acc: {acc:.4f} | "
                f"Pred labels: {unique_preds}"
            )

        if acc == 1.0:
            print("Perfect overfit achieved!")
            break

In [14]:
# model = BiLSTMCRF(
#     embedding_dim=300,
#     hidden_size=256,
#     num_classes=len(label2id),
#     dropout=0
# )

# overfit_one_batch(
#     model,
#     training_data,
#     batch_size=1,
#     epochs=1000,
#     lr=1e-3
# )

In [15]:
model = BiLSTMCRF(
    embedding_dim=300,
    hidden_size=256,
    num_classes=len(label2id),
    dropout=0.3
)

In [16]:
def evaluate(model, val_dataloader, pad_label=-100):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)

    model.eval()

    total_correct = 0
    total_tokens = 0
    total_loss = 0

    with torch.no_grad():

        for x, lengths, y in tqdm(val_dataloader):

            x = x.to(device)
            y = y.to(device)

            mask = (y != pad_label)

            safe_labels = y.clone()
            safe_labels[~mask] = 0

            loss = model.loss(
                x,
                lengths,
                safe_labels,
                mask
            )

            total_loss += loss.item()

            preds = model.decode(
                x,
                lengths,
                mask
            )

            for i in range(len(preds)):

                pred = torch.tensor(
                    preds[i],
                    device=device
                )

                true = safe_labels[i][:len(pred)]

                total_correct += (pred == true).sum().item()
                total_tokens += len(pred)

    val_loss = total_loss / len(val_dataloader)
    val_acc = total_correct / total_tokens

    print(
        f"\nVal Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    return val_loss, val_acc

In [17]:
def train(
    model,
    train_dataset,
    val_dataset,
    batch_size=64,
    epochs=20,
    learning_rate=1e-3,
    pad_label=-100,
    patience=3
):

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=2,
        gamma=0.5
    )

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model = model.to(device)

    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(epochs):

        model.train()

        total_loss = 0
        total_correct = 0
        total_tokens = 0

        for x, lengths, y in tqdm(train_loader):

            x = x.to(device)
            y = y.to(device)
            lengths = lengths.to(device)

            mask = (y != pad_label).bool()

            safe_labels = y.clone()
            safe_labels[~mask] = 0

            optimizer.zero_grad()

            loss = model.loss(
                x,
                lengths,
                safe_labels,
                mask
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5
            )

            optimizer.step()

            total_loss += loss.item()

            with torch.no_grad():

                preds = model.decode(
                    x,
                    lengths,
                    mask
                )

                for i in range(len(preds)):

                    pred = torch.tensor(
                        preds[i],
                        device=device
                    )

                    true = safe_labels[i][mask[i]]

                    total_correct += (pred == true).sum().item()
                    total_tokens += len(true)

        scheduler.step()

        train_loss = total_loss / len(train_loader)
        train_acc = total_correct / total_tokens

        val_loss, val_acc = evaluate(
            model,
            val_loader,
            pad_label
        )

        print(
            f"Epoch {epoch+1} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

        if val_loss < best_val_loss:

            best_val_loss = val_loss
            epochs_no_improve = 0

            torch.save(
                model.state_dict(),
                "bilstm_crf.pth"
            )

        else:

            epochs_no_improve += 1

            if epochs_no_improve >= patience:

                print("Early stopping")

                model.load_state_dict(
                    torch.load(
                        "bilstm_crf.pth",
                        map_location=device
                    )
                )

                break

    return model

In [18]:
trained_model = train(
    model=model,
    train_dataset=training_data,
    val_dataset=val_data,
    batch_size=64,
    epochs=20,
    learning_rate=1e-3,
    pad_label=-100,
    patience=5
)

100%|██████████| 31/31 [00:10<00:00,  3.00it/s]



Val Loss: 4.9321 | Val Acc: 0.9679
Epoch 1 | Train Loss: 16.5713 | Train Acc: 0.9465 | Val Loss: 4.9321 | Val Acc: 0.9679


100%|██████████| 31/31 [00:10<00:00,  2.96it/s]



Val Loss: 2.4007 | Val Acc: 0.9764
Epoch 2 | Train Loss: 3.6543 | Train Acc: 0.9746 | Val Loss: 2.4007 | Val Acc: 0.9764


100%|██████████| 31/31 [00:10<00:00,  2.98it/s]


...


Val Loss: 1.0818 | Val Acc: 0.9863
Epoch 11 | Train Loss: 0.5509 | Train Acc: 0.9911 | Val Loss: 1.0818 | Val Acc: 0.9863


100%|██████████| 31/31 [00:10<00:00,  2.95it/s]



Val Loss: 1.0880 | Val Acc: 0.9862
Epoch 12 | Train Loss: 0.5312 | Train Acc: 0.9916 | Val Loss: 1.0880 | Val Acc: 0.9862


100%|██████████| 31/31 [00:10<00:00,  3.00it/s]


Val Loss: 1.1072 | Val Acc: 0.9869
Epoch 13 | Train Loss: 0.5087 | Train Acc: 0.9918 | Val Loss: 1.1072 | Val Acc: 0.9869
Early stopping


In [19]:
from sklearn.metrics import classification_report, f1_score


def evaluate_test_set(
    model,
    test_dataset,
    label2id,
    batch_size=64,
    pad_label=-100
):

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model = model.to(device)

    model.eval()

    loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    id2label = {
        v: k
        for k, v in label2id.items()
    }

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for x, lengths, y in tqdm(loader):

            x = x.to(device)
            y = y.to(device)

            mask = (y != pad_label)

            preds = model.decode(
                x,
                lengths,
                mask
            )

            for i in range(len(preds)):

                pred = preds[i]

                true = (
                    y[i][mask[i]]
                    .cpu()
                    .tolist()
                )

                all_preds.extend(pred)
                all_labels.extend(true)

    print(
        classification_report(
            all_labels,
            all_preds,
            target_names=[
                id2label[i]
                for i in sorted(id2label)
            ],
            digits=4
        )
    )

    print(
        "Micro F1:",
        f1_score(
            all_labels,
            all_preds,
            average="micro"
        )
    )

    print(
        "Macro F1:",
        f1_score(
            all_labels,
            all_preds,
            average="macro"
        )
    )

In [20]:
evaluate_test_set(
    model=trained_model,
    test_dataset=test_data,
    label2id=label2id,
    batch_size=64,
    pad_label=-100
)

100%|██████████| 34/34 [00:07<00:00,  4.57it/s]


                    precision    recall  f1-score   support

     B-ACCOUNTNAME     0.9709    0.9524    0.9615       105
   B-ACCOUNTNUMBER     1.0000    0.9820    0.9909       111
B-CREDITCARDNUMBER     0.8085    0.8539    0.8306        89
           B-EMAIL     0.9870    0.9744    0.9806       156
            B-IPV4     0.7396    0.9921    0.8475       126
            B-IPV6     0.9222    0.7757    0.8426       107
             B-MAC     1.0000    1.0000    1.0000        84
        B-PASSWORD     0.9694    0.9314    0.9500       102
    B-PHONE_NUMBER     1.0000    0.9907    0.9953       107
             B-SSN     0.9894    1.0000    0.9947        93
        B-USERNAME     0.8154    0.7211    0.7653       147
     I-ACCOUNTNAME     0.9667    0.9721    0.9694       179
   I-ACCOUNTNUMBER     0.9922    0.9897    0.9910       388
I-CREDITCARDNUMBER     0.8030    0.8602    0.8306       758
           I-EMAIL     0.9928    0.9928    0.9928      1258
            I-IPV4     0.7396    0.9921

In [21]:
! pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=43d8dd8aa32ae917cc5412825a82a4c4aa3be0136c95dbe0d6c273c7fd08eade
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [22]:
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch


def evaluate_test_set_seqeval(
    model,
    test_dataset,
    label2id,
    batch_size=64,
    pad_label=-100
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    id2label = {v: k for k, v in label2id.items()}

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for x, lengths, y in tqdm(loader):

            x = x.to(device)
            y = y.to(device)

            mask = (y != pad_label)

            preds = model.decode(x, lengths, mask)

            for i in range(len(preds)):

                pred_ids = preds[i]

                true_ids = y[i][mask[i]].cpu().tolist()

                # convert ids → labels
                pred_labels = [id2label[p] for p in pred_ids]
                true_labels = [id2label[t] for t in true_ids]

                all_preds.append(pred_labels)
                all_labels.append(true_labels)

    # Seqeval reports entity-level metrics
    print(classification_report(all_labels, all_preds, digits=4))

    print("\nMicro F1 (seqeval):", f1_score(all_labels, all_preds))
    print("Precision:", precision_score(all_labels, all_preds))
    print("Recall:", recall_score(all_labels, all_preds))

In [23]:
evaluate_test_set_seqeval(
    model=model,
    test_dataset=test_data,
    label2id=label2id,
    batch_size=64,
    pad_label=-100
)

100%|██████████| 34/34 [00:07<00:00,  4.29it/s]


                  precision    recall  f1-score   support

     ACCOUNTNAME     0.9615    0.9524    0.9569       105
   ACCOUNTNUMBER     0.9909    0.9820    0.9864       111
CREDITCARDNUMBER     0.8000    0.8539    0.8261        89
           EMAIL     0.9806    0.9744    0.9775       156
            IPV4     0.7396    0.9921    0.8475       126
            IPV6     0.8571    0.7290    0.7879       107
             MAC     1.0000    1.0000    1.0000        84
        PASSWORD     0.9091    0.8824    0.8955       102
    PHONE_NUMBER     0.9815    0.9907    0.9860       107
             SSN     0.9787    0.9892    0.9840        93
        USERNAME     0.7710    0.6871    0.7266       147

       micro avg     0.8976    0.9071    0.9023      1227
       macro avg     0.9064    0.9121    0.9068      1227
    weighted avg     0.9015    0.9071    0.9016      1227


Micro F1 (seqeval): 0.9023104985812729
Precision: 0.8975806451612903
Recall: 0.9070904645476773


In [25]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "label2id": label2id,
    "id2label": {v: k for k, v in label2id.items()},
}

torch.save(checkpoint, "pii_ner_model.pth")

print("Model saved successfully as 'pii_ner_model.pth'")

Model saved successfully as 'pii_ner_model.pth'
